# XYZ tiles offline

Build a Dask-backed `xarray.DataArray` from an XYZ config **without** contacting OpenStreetMap.
Tile bytes are synthetic PNGs injected via a mock of `tilearray.array.fetch_tile`.

For live OSM usage (with attribution), see the Quick start page.

In [ ]:
from __future__ import annotations

from io import BytesIO
from unittest.mock import patch

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from tilearray import XYZConfig, create_array
from tilearray.service.xyz import _tile_bounds
from tilearray.types import CRS, BoundingBox, Format, TileRequest, TileResponse

In [ ]:
def make_png_tile(value: int = 128, size: int = 256) -> bytes:
    array = np.full((size, size), value, dtype=np.uint8)
    with BytesIO() as buffer:
        Image.fromarray(array, mode="L").save(buffer, format="PNG")
        return buffer.getvalue()


def single_tile_bbox(zoom: int, tile_x: int, tile_y: int) -> BoundingBox:
    outer = _tile_bounds(tile_x, tile_y, zoom, CRS.EPSG_4326)
    margin_x = (outer.max_x - outer.min_x) * 0.25
    margin_y = (outer.max_y - outer.min_y) * 0.25
    return BoundingBox(
        min_x=outer.min_x + margin_x,
        min_y=outer.min_y + margin_y,
        max_x=outer.max_x - margin_x,
        max_y=outer.max_y - margin_y,
        crs=CRS.EPSG_4326,
    )


png_bytes = make_png_tile(160)


def fake_fetch_tile(request: TileRequest, **kwargs: object) -> TileResponse:
    return TileResponse(
        data=png_bytes,
        content_type="image/png",
        status_code=200,
        headers={},
        url=request.url,
        success=True,
    )

In [ ]:
zoom = 16
tile_x, tile_y = 32768, 21770
bbox = single_tile_bbox(zoom, tile_x, tile_y)

config = XYZConfig.for_openstreetmap(
    zoom=zoom,
    output_format=Format.PNG,
    chunk_size=(256, 256),
)

with patch("tilearray.array.fetch_tile", side_effect=fake_fetch_tile):
    da = create_array(config, bbox, CRS.EPSG_4326, compute=True)

print(da.shape, da.dims, da.attrs["service_type"])
print("mean pixel:", float(da.mean()))
print("fetch policy:", da.attrs.get("tilearray_fetch_policy"))

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
da.plot.imshow(ax=ax, cmap="gray", add_colorbar=False)
ax.set_title("Mocked XYZ tile (offline)")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()